# IT Incident Reporting Automation Using Python

## Project Overview

This project demonstrates the automation of a manual
Excel-based IT incident reporting process using Python.

The solution transforms raw incident data into structured
summary reports and generates a formatted Excel workbook.

## Technologies Used
- Python
- Pandas
- OpenPyXL
- Microsoft Excel

## Automated Reports
1. Priority-wise SLA Analysis (P1–P4)
2. Incident Update Frequency Analysis
3. Monthly Incident Trend Analysis
4. Incident Category Analysis

## Dataset
Synthetic IT incident data generated using ChatGPT.

## Output
An automated Excel report containing Raw Data and Summary sheets.

In [ ]:
import pandas as pd
import numpy as np
import openpyxl
from openpyxl import workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from datetime import datetime

#upload csv
from google.colab import files
uploaded = files.upload()

input_file = next(iter(uploaded))

df = pd.read_csv(input_file)

print("Dataset loaded successfully!")
print("Total Records:", len(df))


In [114]:
df = pd.read_csv('/content/Open_Incident_Raw_Data.csv')
df.head(5)

,Incident Number,Opened Date,Updated Date,Priority,Category,Assignment Group,State,Short Description,Assigned To,Opened By,SLA Status
0,INC07520808,20-09-2026 09:01,15-10-2026 19:08,P2 - High,Hardware,Network Support,In Progress,User reported hardware issue requiring investi...,Priya,System Admin,SLA At Risk
1,INC07520809,24-08-2026 08:01,15-10-2026 19:08,P2 - High,Access,Service Desk,In Progress,User reported access issue requiring investiga...,Rahul,Monitoring Tool,SLA Breached
2,INC07520810,27-08-2026 17:17,15-10-2026 19:08,P3 - Moderate,Hardware,End User Computing,In Progress,User reported hardware issue requiring investi...,Anitha,Service Desk,Within SLA
3,INC07520811,12-07-2026 14:06,15-10-2026 19:08,P3 - Moderate,Cloud,Service Desk,In Progress,User reported cloud issue requiring investigation,Rahul,Service Desk,Within SLA
4,INC07520812,09-09-2026 12:53,15-10-2026 19:08,P4 - Low,Network,Application Support,In Progress,User reported network issue requiring investig...,Divya,End User,Within SLA


In [115]:
df.shape

(1000, 11)

In [116]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Incident Number    1000 non-null   object
 1   Opened Date        1000 non-null   object
 2   Updated Date       1000 non-null   object
 3   Priority           1000 non-null   object
 4   Category           1000 non-null   object
 5   Assignment Group   1000 non-null   object
 6   State              1000 non-null   object
 7   Short Description  1000 non-null   object
 8   Assigned To        1000 non-null   object
 9   Opened By          1000 non-null   object
 10  SLA Status         1000 non-null   object
dtypes: object(11)
memory usage: 86.1+ KB


In [117]:
df.isnull().sum()

,0
Incident Number,0
Opened Date,0
Updated Date,0
Priority,0
Category,0
Assignment Group,0
State,0
Short Description,0
Assigned To,0
Opened By,0


In [118]:
df.duplicated().sum()

np.int64(0)

In [119]:
df['Priority'].unique()

array(['P2 - High', 'P3 - Moderate', 'P4 - Low', 'P1 - Critical'],
      dtype=object)

In [120]:
df['Priority'].value_counts()

,count
Priority,
P3 - Moderate,561
P2 - High,209
P4 - Low,174
P1 - Critical,56


In [121]:
df['Assignment Group'].value_counts()

,count
Assignment Group,
End User Computing,148
Network Support,139
Security Operations,137
Application Support,122
Service Desk,119
Middleware Support,119
Cloud Operations,113
Database Support,103


In [122]:
df['Opened Date'] = pd.to_datetime(df['Opened Date'])
df['Updated Date'] = pd.to_datetime(df['Updated Date'])

/tmp/ipykernel_1309/904359399.py:1: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Opened Date'] = pd.to_datetime(df['Opened Date'])
/tmp/ipykernel_1309/904359399.py:2: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Updated Date'] = pd.to_datetime(df['Updated Date'])


In [123]:
df.columns

Index(['Incident Number', 'Opened Date', 'Updated Date', 'Priority',
       'Category', 'Assignment Group', 'State', 'Short Description',
       'Assigned To', 'Opened By', 'SLA Status'],
      dtype='object')

In [124]:
days = (pd.Timestamp.today().normalize() - df['Updated Date'].dt.normalize()).dt.days

def update_range(days):
    if days <= 1:
        return 'Within a day'
    elif days <= 2:
        return '1 to 2 days'
    elif days <= 5:
        return '2 to 5 days'
    elif days <= 10:
        return '5 to 10 days'
    elif days <= 20:
        return '10 to 20 days'
    else:
        return '>= 30 days'

df['Update Frequency Range'] = days.apply(update_range)
df[['Opened Date','Updated Date', 'Update Frequency Range']].head()




,Opened Date,Updated Date,Update Frequency Range
0,2026-09-20 09:01:00,2026-10-15 19:08:00,10 to 20 days
1,2026-08-24 08:01:00,2026-10-15 19:08:00,10 to 20 days
2,2026-08-27 17:17:00,2026-10-15 19:08:00,10 to 20 days
3,2026-07-12 14:06:00,2026-10-15 19:08:00,10 to 20 days
4,2026-09-09 12:53:00,2026-10-15 19:08:00,10 to 20 days


In [125]:
# Open Incident SLA

for priority in sorted(df['Priority'].dropna().unique()):

    priority_data = df[df['Priority'] == priority]

    table = pd.crosstab(
        priority_data['Assignment Group'],
        priority_data['SLA Status']
    )

    # Keep columns in required order
    table = table.reindex(
        columns=['Within SLA', 'SLA At Risk', 'SLA Breached'],
        fill_value=0
    )

    # Add Total
    table['Total'] = table.sum(axis=1)

    # Add Grand Total
    table.loc['Grand Total'] = table.sum()

    # Convert Assignment Group from index to normal column
    table = table.reset_index()

    # Make sure header is clean
    table.columns.name = None

    if 'sla_tables' not in locals():
        sla_tables = {}

    sla_tables[priority] = table.copy()

    print(f"\nOpen Incident SLA {priority}")
    display(table.style.hide(axis="index")) #table to hide unwanted column name

    # Hide index numbers
display(table.style.hide(axis="index"))



Open Incident SLA P1 - Critical


Assignment Group,Within SLA,SLA At Risk,SLA Breached,Total
Application Support,5,0,0,5
Cloud Operations,7,0,3,10
Database Support,3,0,2,5
End User Computing,4,1,3,8
Middleware Support,6,2,1,9
Network Support,3,1,0,4
Security Operations,5,1,2,8
Service Desk,5,1,1,7
Grand Total,38,6,12,56



Open Incident SLA P2 - High


Assignment Group,Within SLA,SLA At Risk,SLA Breached,Total
Application Support,12,4,11,27
Cloud Operations,19,3,4,26
Database Support,10,8,3,21
End User Computing,14,5,6,25
Middleware Support,17,2,5,24
Network Support,21,5,9,35
Security Operations,19,5,5,29
Service Desk,15,3,4,22
Grand Total,127,35,47,209



Open Incident SLA P3 - Moderate


Assignment Group,Within SLA,SLA At Risk,SLA Breached,Total
Application Support,43,10,23,76
Cloud Operations,42,7,8,57
Database Support,36,10,10,56
End User Computing,55,8,22,85
Middleware Support,46,7,15,68
Network Support,48,12,19,79
Security Operations,48,9,14,71
Service Desk,49,15,5,69
Grand Total,367,78,116,561



Open Incident SLA P4 - Low


Assignment Group,Within SLA,SLA At Risk,SLA Breached,Total
Application Support,11,2,1,14
Cloud Operations,14,2,4,20
Database Support,14,3,4,21
End User Computing,15,2,13,30
Middleware Support,12,2,4,18
Network Support,12,5,4,21
Security Operations,18,4,7,29
Service Desk,13,3,5,21
Grand Total,109,23,42,174


Assignment Group,Within SLA,SLA At Risk,SLA Breached,Total
Application Support,11,2,1,14
Cloud Operations,14,2,4,20
Database Support,14,3,4,21
End User Computing,15,2,13,30
Middleware Support,12,2,4,18
Network Support,12,5,4,21
Security Operations,18,4,7,29
Service Desk,13,3,5,21
Grand Total,109,23,42,174


In [126]:
# Update Frequency Pivot Table

table = pd.crosstab(
    df['Assignment Group'],
    df['Update Frequency Range']
)

# Keep ranges in required order
table = table.reindex(
    columns=[
        'Within a day',
        '1 to 2 days',
        '2 to 5 days',
        '5 to 10 days',
        '10 to 20 days',
        '>= 30 days'
    ],
    fill_value=0
)

# Add Total column
table['Total'] = table.sum(axis=1)

# Add Grand Total row
table.loc['Grand Total'] = table.sum()

# Convert Assignment Group to normal column
table = table.reset_index()

# Remove extra column name
table.columns.name = None

# Store final table
update_frequency_table = table.copy()

# Hide index numbers
display(table.style.hide(axis="index"))

Assignment Group,Within a day,1 to 2 days,2 to 5 days,5 to 10 days,10 to 20 days,>= 30 days,Total
Application Support,3,2,0,75,40,2,122
Cloud Operations,3,0,0,78,32,0,113
Database Support,0,0,0,75,26,2,103
End User Computing,4,1,0,99,40,4,148
Middleware Support,0,1,0,80,38,0,119
Network Support,3,2,0,87,46,1,139
Security Operations,6,4,0,80,47,0,137
Service Desk,3,1,0,79,35,1,119
Grand Total,22,11,0,653,304,10,1000


In [127]:
# Monthly incident Trend
# Create Month column
df['Month'] = df['Opened Date'].dt.strftime('%B')

# Create pivot table
table = pd.crosstab(
    df['Assignment Group'],
    df['Month']
)

# Add Total
table['Total'] = table.sum(axis=1)

# Add Grand Total
table.loc['Grand Total'] = table.sum()

# Convert Assignment Group to normal column
table = table.reset_index()

# Remove extra column name
table.columns.name = None

monthly_table = table.copy()

print("Open Incident Monthly Trend")

# Hide index numbers
display(table.style.hide(axis="index"))

Open Incident Monthly Trend


Assignment Group,August,July,October,September,Total
Application Support,36,42,3,41,122
Cloud Operations,44,27,4,38,113
Database Support,31,29,5,38,103
End User Computing,44,55,2,47,148
Middleware Support,38,42,3,36,119
Network Support,34,41,2,62,139
Security Operations,40,41,4,52,137
Service Desk,38,34,7,40,119
Grand Total,305,311,30,354,1000


In [128]:
#Category Analysis

table = df['Category'].value_counts().reset_index()

# Rename columns
table.columns = ['Category', 'Incident Count']

print("Open Incident Category Analysis")

# Add Grand Total
grand_total = table['Incident Count'].sum()

table.loc[len(table)] = ['Grand Total', grand_total]

category_table = table.copy()

# Hide index numbers
display(table.style.hide(axis="index"))

Open Incident Category Analysis


Category,Incident Count
Email,159
Database,148
Access,146
Network,142
Software,141
Cloud,135
Hardware,129
Grand Total,1000


In [131]:
# 1. FILE NAME WITH TODAY'S DATE

today = datetime.today().strftime('%Y-%m-%d')

file_name = f'Open_Incident_{today}.xlsx'


# 2. CREATE EXCEL FILE

with pd.ExcelWriter(file_name, engine='openpyxl') as writer:

    # Raw Data - Sheet 2
    df.to_excel(
        writer,
        sheet_name='Raw Data',
        index=False
    )

    # Summary - Sheet 1
    pd.DataFrame().to_excel(
        writer,
        sheet_name='Summary',
        index=False
    )


# Open workbook
wb = load_workbook(file_name)

ws = wb['Summary']

# 3. FORMATTING STYLES

title_fill = PatternFill(
    'solid',
    fgColor='4472C4'
)

title_font = Font(
    bold=True,
    color='FFFFFF',
    size=14
)

header_fill = PatternFill(
    'solid',
    fgColor='D9EAF7'
)

header_font = Font(
    bold=True
)

grand_total_fill = PatternFill(
    'solid',
    fgColor='F2F2F2'
)

thin = Side(style='thin')

border = Border(
    left=thin,
    right=thin,
    top=thin,
    bottom=thin
)


# 4. FUNCTION TO WRITE + FORMAT A TABLE

def write_table(ws, table, start_row, title):

    # ----- Title -----
    title_cell = ws.cell(
        row=start_row,
        column=1
    )

    title_cell.value = title
    title_cell.font = title_font
    title_cell.fill = title_fill
    title_cell.alignment = Alignment(
        horizontal='center'
    )

    # Number of columns
    number_of_columns = len(table.columns)

    # Merge title
    ws.merge_cells(
        start_row=start_row,
        start_column=1,
        end_row=start_row,
        end_column=number_of_columns
    )

    # Header
    header_row = start_row + 1

    for col_num, column_name in enumerate(
        table.columns,
        1

    ):

        cell = ws.cell(
            row=header_row,
            column=col_num
        )

        cell.value = column_name
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(
            horizontal='center'
        )
        cell.border = border

    #Data
    for row_num, row_data in enumerate(
        table.itertuples(index=False),
        header_row + 1
    ):

        for col_num, value in enumerate(
            row_data,
            1
        ):

            cell = ws.cell(
                row=row_num,
                column=col_num
            )

            cell.value = value
            cell.alignment = Alignment(
                horizontal='center'
            )
            cell.border = border

    # Format Grand Total row
    grand_total_row = header_row + len(table)
    if ws.cell(row=grand_total_row, column=1).value == 'Grand Total':
        for col_num in range(1, number_of_columns + 1):
            cell = ws.cell(row=grand_total_row, column=col_num)
            cell.font = Font(bold=True)
            cell.fill = grand_total_fill
            cell.border = border
            cell.alignment = Alignment(horizontal='center')

    # Return the next starting row with some padding
    return grand_total_row + 3

# 5. WRITE THE 4 TABLES

row = 1

# ----- Open Incident SLA -----

for priority in sorted(sla_tables.keys()):

    row = write_table(
        ws,
        sla_tables[priority],
        row,
        f'Open Incident SLA {priority}'
    )


#Update Frequency

row = write_table(
    ws,
    update_frequency_table,
    row,
    'Open Incident Update Frequency'
)


#Monthly Incident Trend

row = write_table(
    ws,
    monthly_table,
    row,
    'Open Incident Monthly Trend'
)


# Category Analysis

ow = write_table(
    ws,
    category_table,
    row,
    'Open Incident Category Analysis'
)


# 6. COLUMN WIDTH

for col_num in range(
    1,
    ws.max_column + 1
):

    max_length = 0

    for row_num in range(
        1,
        ws.max_row + 1
    ):

        value = ws.cell(
            row=row_num,
            column=col_num
        ).value

        if value is not None:

            max_length = max(
                max_length,
                len(str(value))
            )

    ws.column_dimensions[
        get_column_letter(col_num)
    ].width = min(
        max_length + 3,
        30
    )


# 7. RAW DATA FORMATTING

raw_ws = wb['Raw Data']

# Freeze header
raw_ws.freeze_panes = 'A2'

# Filter
raw_ws.auto_filter.ref = raw_ws.dimensions

# Header formatting
for cell in raw_ws[1]:

    cell.font = Font(bold=True)

    cell.fill = header_fill

    cell.border = border

    cell.alignment = Alignment(horizontal='center')

    cell.border = border


# Raw data column width
for col_num in range(1,raw_ws.max_column + 1):

    raw_ws.column_dimensions[get_column_letter(col_num)].width = 18

# 8. SAVE

wb.save(file_name)

print(f"Excel report created: {file_name}")

print("Dataset Validation")
print("------------------")

print("Total Records:", len(df))
print("Total Columns:", len(df.columns))

print("Duplicate Records:", df.duplicated().sum())


Excel report created: Open_Incident_2026-10-28.xlsx
Dataset Validation
------------------
Total Records: 1000
Total Columns: 13
Duplicate Records: 0


## Conclusion

This project successfully demonstrates the automation of a manual Excel-based IT incident reporting process using **Python, Pandas, and OpenPyXL**.

### Key Achievements
- Automated data cleaning and preprocessing.
- Generated priority-wise SLA reports (P1–P4).
- Automated incident update frequency calculations.
- Analyzed monthly incident trends.
- Generated category-wise incident summaries.
- Created a formatted Excel report with multiple summary tables, headers, borders, and grand totals.
- Reduced the need for repetitive manual Excel operations.

### Project Outcome

The automation processes **1,000 synthetic incident records** and generates a structured Excel workbook containing Raw Data and Summary sheets.

### Business Impact

This solution demonstrates how Python automation can simplify repetitive reporting tasks, improve reporting consistency, and reduce manual effort.

### Future Enhancements
- Automate daily report execution.
- Develop an interactive Power BI dashboard.
- Integrate with a database for automated data retrieval.
- Add automated report distribution through email.

**Overall, this project demonstrates practical skills in Python automation, data analysis, data transformation, and Excel reporting.**